# demo_1 vs demo2 — CatBoost·XGBoost 비교 및 최종 결론

## 결론부터

프로젝트 서비스가 **'매주 다음 주 이탈 위험자를 갱신'**하는 것이라면, 현재 그대로는 **demo_1의 주차별 데이터 구조 + CatBoost**가 가장 적합합니다.

- demo_1은 1~38주차를 매주 다루므로 서비스 주기와 맞습니다.
- 두 버전 모두 CatBoost가 PR-AUC와 확률 품질에서 더 균형 잡혔습니다.
- demo2는 Feature가 더 풍부하고 검증·모델 저장이 잘 되어 있지만 현재 비교 데이터가 4·19·25주차뿐입니다.

장기적으로 가장 좋은 조합은 **demo_1의 전체 주차 행 구조 + demo2의 확장 Feature·누수 검사·확률 보정 + CatBoost**입니다.


## 1. 데이터와 Target 차이

| 구분 | demo_1 | demo2 비교본 |
|---|---:|---:|
| 행 수 | 895,005 | 75,125 |
| 다음 주 이탈 건수 | 6,672 | 588 |
| 이탈률 | 0.7455% | 0.7827% |
| 예측 주차 | 1~38주차 | 4·19·25주차 |
| Feature 수 | 51 | 98 |
| Target | 다음 주 이탈 | 다음 7일 이내 이탈 |

두 데이터는 행의 범위가 다르므로 demo_1의 PR-AUC와 demo2의 PR-AUC를 숫자만 보고 직접 우열 비교하면 안 됩니다. **각 데이터 안에서 XGBoost와 CatBoost를 비교**해야 합니다.


## 2. Feature 구성은 무엇이 다른가?

### demo_1 — 단순하고 넓은 주차 범위

- 학생 기본정보
- 등록 시점
- 기준 주차까지의 누적 VLE 클릭·활동
- 기준 주차까지의 누적 평가 제출·점수
- 장점: 51개로 비교적 단순하고 1~38주 전체에 적용됨
- 단점: 최근 한 주 급감, 이전 주 대비 변화, 활동 중단 기간 같은 변화 신호가 상대적으로 적음

### demo2 — 최근 변화까지 자세히 표현

demo_1 유형의 기본·누적 Feature에 더해 다음을 포함합니다.

- 현재 주차와 이전 주차 활동
- 클릭·활동일·콘텐츠 수 증감
- 마지막 활동 이후 지난 주 수
- 활동 유형별 클릭 비율
- 클릭 수 로그 변환
- 평가 유형(TMA·CMA·Exam)별 제출 현황
- 미제출·지각·점수 관련 세부 지표

- 장점: 최근 행동 변화를 더 잘 표현함
- 단점: Feature 수가 많아 복잡하고, 현재는 세 주차만 있어 매주 서비스에 바로 쓰기 어려움

> demo_1의 `used_data/weekly_next_week_with_vle.csv`와 OOF 파일은 해당 브랜치에 포함되어 있지 않습니다. 따라서 아래 demo_1 목록은 노트북에 문서화된 Feature이며, 정확한 51개 열 전체 이름은 현재 GitHub 파일만으로 검증할 수 없습니다.


## 3. demo_1에서 문서로 확인되는 Feature

| group | documented_name | beginner_description | exact_51_column_list_verified |
|---|---|---|---|
| 과목·운영 | code_module | 과목 코드 | 아니오 |
| 과목·운영 | code_presentation | 과목 운영 학기·회차 | 아니오 |
| 과목·운영 | prediction_week | 예측 기준 주차 | 아니오 |
| 과목·운영 | cutoff_day | 예측 기준 주차의 마지막 날짜 | 아니오 |
| 과목·운영 | module_presentation_length | 강좌 운영 기간 | 아니오 |
| 학생 기본정보 | gender | 성별 | 아니오 |
| 학생 기본정보 | region | 거주 지역 | 아니오 |
| 학생 기본정보 | highest_education | 최종 학력 | 아니오 |
| 학생 기본정보 | imd_band | 사회경제적 불리함 구간 | 아니오 |
| 학생 기본정보 | age_band | 연령대 | 아니오 |
| 학생 기본정보 | num_of_prev_attempts | 이전 수강 시도 횟수 | 아니오 |
| 학생 기본정보 | studied_credits | 신청 학점 | 아니오 |
| 학생 기본정보 | disability | 장애 여부 | 아니오 |
| 등록정보 | date_registration | 등록일 | 아니오 |
| 등록정보 | registered_after_start | 개강 후 등록 여부 | 아니오 |
| 등록정보 | registration_lead_days | 개강 전 미리 등록한 일수 | 아니오 |
| 등록정보 | late_registration_days | 개강 후 늦게 등록한 일수 | 아니오 |
| 평가 누적 | documented: assessment count | 기준일까지의 평가 개수 | 아니오 |
| 평가 누적 | documented: assessment weight | 기준일까지의 평가 비중 | 아니오 |
| 평가 누적 | documented: submitted count | 기준일까지 제출한 평가 개수 | 아니오 |
| 평가 누적 | documented: late count | 지각 제출 개수 | 아니오 |
| 평가 누적 | documented: missing count | 미제출 평가 개수 | 아니오 |
| 평가 누적 | documented: submission rate | 평가 제출률 | 아니오 |
| 평가 누적 | documented: mean score | 평균 평가 점수 | 아니오 |
| VLE 누적 | documented: total clicks | 기준일까지의 전체 클릭 수 | 아니오 |
| VLE 누적 | documented: interaction rows | 기준일까지의 VLE 로그 행 수 | 아니오 |
| VLE 누적 | documented: last activity day | 마지막 VLE 활동일 | 아니오 |
| VLE 누적 | documented: active days | 누적 활동일 수 | 아니오 |
| VLE 누적 | documented: unique sites | 누적 고유 콘텐츠 수 | 아니오 |
| VLE 누적 | documented: activity type clicks | 활동 유형별 누적 클릭 수 | 아니오 |
| VLE 누적 | documented: VLE record flag | VLE 활동 기록 존재 여부 | 아니오 |

## 4. demo2 전체 열과 각 Feature 설명

`used_by_model=아니오`인 `id_student`는 분할 전용이고 Target은 정답입니다. 나머지 98개가 모델 입력입니다.

| column_order | column_name | role | group | beginner_description | used_by_model |
|---|---|---|---|---|---|
| 1 | code_module | 모델 Feature | 과목·운영 | 수강 과목을 구분하는 코드입니다. | 예 |
| 2 | code_presentation | 모델 Feature | 과목·운영 | 같은 과목이 어느 학기·운영회차에 열렸는지 나타냅니다. | 예 |
| 3 | id_student | 학생 분할 전용 | 식별자 | 학생 식별자입니다. 모델 입력에는 넣지 않고 학생 단위 데이터 분할에만 사용합니다. | 아니오 |
| 4 | prediction_week | 모델 Feature | 과목·운영 | 이 행에서 위험도를 계산하는 기준 주차입니다. | 예 |
| 5 | cutoff_day | 모델 Feature | 과목·운영 | 기준 주차의 마지막 날짜입니다. prediction_week×7-1로 계산합니다. | 예 |
| 6 | module_presentation_length | 모델 Feature | 과목·운영 | 해당 강좌 운영 기간을 일수로 나타냅니다. | 예 |
| 7 | cum_total_clicks | 모델 Feature | VLE 누적 | 기준 주차까지 발생한 전체 VLE 클릭 수입니다. | 예 |
| 8 | cum_interaction_rows | 모델 Feature | VLE 누적 | 기준 주차까지 쌓인 VLE 활동 로그 행 수입니다. | 예 |
| 9 | cum_active_days | 모델 Feature | VLE 누적 | 기준 주차까지 VLE를 사용한 서로 다른 날짜 수입니다. | 예 |
| 10 | cum_unique_site_week_count | 모델 Feature | VLE 누적 | 주차별 고유 콘텐츠 이용 수를 기준 주차까지 누적한 값입니다. | 예 |
| 11 | cum_activity_type_week_count | 모델 Feature | VLE 누적 | 주차별 이용 활동 유형 수를 기준 주차까지 누적한 값입니다. | 예 |
| 12 | cum_forumng_clicks | 모델 Feature | VLE 누적 | 기준 주차까지 발생한 포럼 클릭 수입니다. | 예 |
| 13 | cum_quiz_clicks | 모델 Feature | VLE 누적 | 기준 주차까지 발생한 퀴즈 클릭 수입니다. | 예 |
| 14 | cum_oucontent_clicks | 모델 Feature | VLE 누적 | 기준 주차까지 발생한 강의 콘텐츠 클릭 수입니다. | 예 |
| 15 | cum_resource_clicks | 모델 Feature | VLE 누적 | 기준 주차까지 발생한 학습 자료 클릭 수입니다. | 예 |
| 16 | cum_other_clicks | 모델 Feature | VLE 누적 | 기준 주차까지 발생한 기타 활동 클릭 수입니다. | 예 |
| 17 | observed_weeks | 모델 Feature | VLE 참여 | 등록 후 기준 주차까지 관찰된 주차 수입니다. | 예 |
| 18 | active_weeks | 모델 Feature | VLE 참여 | 관찰 주차 중 VLE 활동 기록이 한 번이라도 있었던 주차 수입니다. | 예 |
| 19 | pre_course_clicks | 모델 Feature | VLE 참여 | 강좌 시작 전 발생한 총 클릭 수입니다. | 예 |
| 20 | pre_course_interaction_rows | 모델 Feature | VLE 참여 | 강좌 시작 전 VLE 로그 행 수입니다. | 예 |
| 21 | inactive_weeks | 모델 Feature | VLE 참여 | 관찰 주차 중 활동 기록이 없었던 주차 수입니다. | 예 |
| 22 | active_week_rate | 모델 Feature | VLE 참여 | 관찰 주차 중 활동한 주차의 비율입니다. | 예 |
| 23 | cum_avg_clicks_per_active_day | 모델 Feature | VLE 누적 | 기준 주차까지 활동한 날 하루당 평균 클릭 수입니다. | 예 |
| 24 | cum_avg_clicks_per_site_week | 모델 Feature | VLE 누적 | 기준 주차까지 콘텐츠-주차 단위당 평균 클릭 수입니다. | 예 |
| 25 | current_total_clicks | 모델 Feature | VLE 현재 | 현재 주차의 전체 클릭 수입니다. | 예 |
| 26 | current_interaction_rows | 모델 Feature | VLE 현재 | 현재 주차의 VLE 로그 행 수입니다. | 예 |
| 27 | current_active_days | 모델 Feature | VLE 현재 | 현재 주차에 활동한 날짜 수입니다. | 예 |
| 28 | current_unique_sites | 모델 Feature | VLE 현재 | 현재 주차에 이용한 고유 콘텐츠 수입니다. | 예 |
| 29 | current_activity_type_count | 모델 Feature | VLE 현재 | 현재 주차에 이용한 활동 유형 수입니다. | 예 |
| 30 | current_forumng_clicks | 모델 Feature | VLE 현재 | 현재 주차의 포럼 클릭 수입니다. | 예 |
| 31 | current_quiz_clicks | 모델 Feature | VLE 현재 | 현재 주차의 퀴즈 클릭 수입니다. | 예 |
| 32 | current_oucontent_clicks | 모델 Feature | VLE 현재 | 현재 주차의 강의 콘텐츠 클릭 수입니다. | 예 |
| 33 | current_resource_clicks | 모델 Feature | VLE 현재 | 현재 주차의 학습 자료 클릭 수입니다. | 예 |
| 34 | current_other_clicks | 모델 Feature | VLE 현재 | 현재 주차의 기타 활동 클릭 수입니다. | 예 |
| 35 | current_has_vle_record | 모델 Feature | VLE 현재 | 현재 주차에 VLE 활동 기록이 있으면 1, 없으면 0입니다. | 예 |
| 36 | previous_total_clicks | 모델 Feature | VLE 이전 | 이전 주차의 전체 클릭 수입니다. | 예 |
| 37 | previous_active_days | 모델 Feature | VLE 이전 | 이전 주차에 활동한 날짜 수입니다. | 예 |
| 38 | previous_unique_sites | 모델 Feature | VLE 이전 | 이전 주차에 이용한 고유 콘텐츠 수입니다. | 예 |
| 39 | last_active_week | 모델 Feature | VLE 최근성 | 가장 마지막으로 VLE 활동이 있었던 주차입니다. | 예 |
| 40 | weeks_since_last_activity | 모델 Feature | VLE 최근성 | 마지막 활동 이후 현재까지 몇 주가 지났는지 나타냅니다. | 예 |
| 41 | current_no_activity | 모델 Feature | VLE 현재 | 현재 주차에 활동이 전혀 없으면 1입니다. | 예 |
| 42 | click_change | 모델 Feature | VLE 변화 | 현재 주차 클릭 수에서 이전 주차 클릭 수를 뺀 값입니다. | 예 |
| 43 | click_change_rate | 모델 Feature | VLE 변화 | 이전 주차 대비 현재 클릭 수의 증감률입니다. | 예 |
| 44 | active_days_change | 모델 Feature | VLE 변화 | 현재 주차 활동일 수와 이전 주차 활동일 수의 차이입니다. | 예 |
| 45 | unique_sites_change | 모델 Feature | VLE 변화 | 현재 주차와 이전 주차의 이용 콘텐츠 수 차이입니다. | 예 |
| 46 | cum_forumng_share | 모델 Feature | VLE 구성비 | 누적 전체 클릭 중 포럼 클릭이 차지하는 비율입니다. | 예 |
| 47 | cum_quiz_share | 모델 Feature | VLE 구성비 | 누적 전체 클릭 중 퀴즈 클릭이 차지하는 비율입니다. | 예 |
| 48 | cum_oucontent_share | 모델 Feature | VLE 구성비 | 누적 전체 클릭 중 강의 콘텐츠 클릭이 차지하는 비율입니다. | 예 |
| 49 | cum_resource_share | 모델 Feature | VLE 구성비 | 누적 전체 클릭 중 학습 자료 클릭이 차지하는 비율입니다. | 예 |
| 50 | cum_other_share | 모델 Feature | VLE 구성비 | 누적 전체 클릭 중 기타 활동 클릭이 차지하는 비율입니다. | 예 |
| 51 | log1p_cum_total_clicks | 모델 Feature | VLE 변환 | 누적 전체 클릭 수에 log(1+x)를 적용해 큰 값의 영향을 줄인 Feature입니다. | 예 |
| 52 | log1p_current_total_clicks | 모델 Feature | VLE 변환 | 현재 주차 클릭 수에 log(1+x)를 적용해 큰 값의 영향을 줄인 Feature입니다. | 예 |
| 53 | log1p_pre_course_clicks | 모델 Feature | VLE 변환 | 강좌 시작 전 클릭 수에 log(1+x)를 적용해 큰 값의 영향을 줄인 Feature입니다. | 예 |
| 54 | gender | 모델 Feature | 학생 기본정보 | 학생의 성별 범주입니다. | 예 |
| 55 | region | 모델 Feature | 학생 기본정보 | 학생이 거주하는 지역 범주입니다. | 예 |
| 56 | highest_education | 모델 Feature | 학생 기본정보 | 학생의 최종 학력 범주입니다. | 예 |
| 57 | imd_band | 모델 Feature | 학생 기본정보 | 거주 지역의 사회경제적 불리함 정도를 구간으로 표현한 값입니다. | 예 |
| 58 | imd_band_missing | 모델 Feature | 학생 기본정보 | IMD 구간 정보가 원래 비어 있었으면 1입니다. | 예 |
| 59 | age_band | 모델 Feature | 학생 기본정보 | 학생의 연령대 범주입니다. | 예 |
| 60 | num_of_prev_attempts | 모델 Feature | 학생 기본정보 | 같은 과목을 이전에 수강한 횟수입니다. | 예 |
| 61 | studied_credits | 모델 Feature | 학생 기본정보 | 해당 학기에 신청한 총 학점입니다. | 예 |
| 62 | disability | 모델 Feature | 학생 기본정보 | 장애 여부 범주입니다. | 예 |
| 63 | date_registration_missing | 모델 Feature | 등록정보 | 등록일 정보가 원래 비어 있었으면 1입니다. | 예 |
| 64 | registered_after_start | 모델 Feature | 등록정보 | 강좌 시작 후 등록했으면 1입니다. | 예 |
| 65 | registration_lead_days | 모델 Feature | 등록정보 | 강좌 시작 전에 미리 등록한 일수입니다. | 예 |
| 66 | late_registration_days | 모델 Feature | 등록정보 | 강좌 시작 후 늦게 등록한 일수입니다. | 예 |
| 67 | assessment_due_count | 모델 Feature | 평가 | 기준일까지 마감된 평가의 개수입니다. | 예 |
| 68 | assessment_due_weight | 모델 Feature | 평가 | 기준일까지 마감된 평가 비중의 합입니다. | 예 |
| 69 | assessment_submitted_due_count | 모델 Feature | 평가 | 마감된 평가 중 제출한 개수입니다. | 예 |
| 70 | assessment_scored_due_count | 모델 Feature | 평가 | 마감된 평가 중 점수가 확인된 개수입니다. | 예 |
| 71 | assessment_missing_score_count | 모델 Feature | 평가 | 제출 정보는 있지만 점수가 없는 평가 개수입니다. | 예 |
| 72 | assessment_banked_due_count | 모델 Feature | 평가 | 이전 결과가 인정되어 banked 처리된 평가 개수입니다. | 예 |
| 73 | assessment_late_count | 모델 Feature | 평가 | 마감일보다 늦게 제출한 평가 개수입니다. | 예 |
| 74 | assessment_nonbanked_submitted_count | 모델 Feature | 평가 | banked가 아닌 평가 중 실제 제출한 개수입니다. | 예 |
| 75 | assessment_mean_score | 모델 Feature | 평가 | 기준일까지 확인된 평가 점수의 평균입니다. | 예 |
| 76 | assessment_median_score | 모델 Feature | 평가 | 기준일까지 확인된 평가 점수의 중앙값입니다. | 예 |
| 77 | assessment_min_score | 모델 Feature | 평가 | 기준일까지 확인된 평가 점수의 최솟값입니다. | 예 |
| 78 | assessment_max_score | 모델 Feature | 평가 | 기준일까지 확인된 평가 점수의 최댓값입니다. | 예 |
| 79 | weighted_score_sum | 모델 Feature | 평가 | 점수×평가 비중을 모두 더한 값입니다. | 예 |
| 80 | scored_weight_sum | 모델 Feature | 평가 | 점수가 확인된 평가들의 비중 합입니다. | 예 |
| 81 | assessment_mean_submission_gap | 모델 Feature | 평가 | 제출일-마감일 차이의 평균입니다. 음수면 일찍 제출한 것입니다. | 예 |
| 82 | assessment_median_submission_gap | 모델 Feature | 평가 | 제출일-마감일 차이의 중앙값입니다. | 예 |
| 83 | assessment_due_tma_count | 모델 Feature | 평가 유형 | 기준일까지 마감된 TMA 평가 개수입니다. | 예 |
| 84 | assessment_submitted_tma_count | 모델 Feature | 평가 유형 | 마감된 TMA 중 제출한 개수입니다. | 예 |
| 85 | assessment_due_cma_count | 모델 Feature | 평가 유형 | 기준일까지 마감된 CMA 평가 개수입니다. | 예 |
| 86 | assessment_submitted_cma_count | 모델 Feature | 평가 유형 | 마감된 CMA 중 제출한 개수입니다. | 예 |
| 87 | assessment_due_exam_count | 모델 Feature | 평가 유형 | 기준일까지 마감된 Exam 개수입니다. | 예 |
| 88 | assessment_submitted_exam_count | 모델 Feature | 평가 유형 | 마감된 Exam 중 제출한 개수입니다. | 예 |
| 89 | assessment_missing_due_count | 모델 Feature | 평가 | 마감됐지만 제출하지 않은 평가 개수입니다. | 예 |
| 90 | assessment_submission_rate | 모델 Feature | 평가 | 마감된 평가 중 제출한 비율입니다. | 예 |
| 91 | assessment_missing_due_rate | 모델 Feature | 평가 | 마감된 평가 중 미제출 비율입니다. | 예 |
| 92 | assessment_late_rate | 모델 Feature | 평가 | 제출한 평가 중 지각 제출 비율입니다. | 예 |
| 93 | assessment_weighted_mean_score | 모델 Feature | 평가 | 평가 비중을 반영한 평균 점수입니다. | 예 |
| 94 | any_known_submission_count | 모델 Feature | 평가 보조 | 기준일까지 제출 사실을 알 수 있는 전체 평가 개수입니다. | 예 |
| 95 | any_known_scored_count | 모델 Feature | 평가 보조 | 기준일까지 점수를 알 수 있는 전체 평가 개수입니다. | 예 |
| 96 | any_known_score_missing_count | 모델 Feature | 평가 보조 | 기준일까지 제출은 확인되지만 점수가 없는 전체 평가 개수입니다. | 예 |
| 97 | any_known_banked_count | 모델 Feature | 평가 보조 | 기준일까지 banked로 확인된 전체 평가 개수입니다. | 예 |
| 98 | any_known_mean_score | 모델 Feature | 평가 보조 | 마감 여부와 관계없이 기준일까지 알려진 점수의 평균입니다. | 예 |
| 99 | any_known_median_score | 모델 Feature | 평가 보조 | 마감 여부와 관계없이 기준일까지 알려진 점수의 중앙값입니다. | 예 |
| 100 | target_next_week_withdrawn | 정답 | 정답 | 기준 주차 다음 7일 안에 이탈했으면 1, 아니면 0입니다. | 아니오 |

## 5. 같은 데이터 안에서 모델 성능 비교

### demo_1 결과

| 모델 | Recall@Top-20% ↑ | PR-AUC ↑ | Brier ↓ | ECE ↓ |
|---|---:|---:|---:|---:|
| XGBoost | **0.6993** | 0.0808 | 0.148658 | 0.299222 |
| CatBoost | 0.6962 | **0.0893** | **0.007066** | **0.000198** |

### demo2 결과

| 모델 | Recall@Top-20% ↑ | PR-AUC ↑ | Brier ↓ | ECE ↓ |
|---|---:|---:|---:|---:|
| XGBoost | **0.6990** | 0.0396 | 0.113504 | 0.263375 |
| CatBoost | 0.6837 | **0.0438** | **0.007615** | **0.000150** |

두 버전에서 공통으로 나타난 패턴은 같습니다.

- XGBoost는 위험 상위 20% Recall이 조금 높습니다.
- CatBoost는 PR-AUC가 더 높고 확률 오차가 훨씬 낮습니다.
- XGBoost는 `scale_pos_weight` 때문에 원시 확률이 크게 부풀었습니다.


## 6. 장점과 단점

| 선택지 | 장점 | 단점 |
|---|---|---|
| demo_1 XGBoost | 1~38주 매주 예측, 상위 위험군 Recall이 근소하게 높음 | One-Hot 필요, 원시 확률이 심하게 부풀어 화면 표시 부적합, 모델 파일 미포함 |
| demo_1 CatBoost | 1~38주 매주 예측, PR-AUC·확률 품질 우수, 범주형 처리 간단 | 최종 저장 모델·확률 보정기가 브랜치에 없음, Test Fold를 조기 종료에도 사용 |
| demo2 XGBoost | 98개 상세 Feature, 모델·전처리기 저장, 재현 가능 | 4·19·25주만 지원, One-Hot 필요, 원시 확률 보정 불량 |
| demo2 CatBoost | 98개 상세 Feature, 범주형 직접 처리, 확률 품질 우수, 기존 Platt 검증 경험 있음 | 현재 세 주차만 지원하여 매주 서비스 불가, 비교용 모델에는 아직 전 주차 데이터가 없음 |


## 7. GitHub 프로젝트 주제에 적합한 선택

GitHub README의 큰 목표는 **OULAD 중도이탈 조기예측과 적절한 개입 시점 제안**입니다. 또한 현재 논의한 서비스 목표는 **매주 다음 주 이탈 위험자 갱신**입니다.

### 기존 파일 중 하나만 고르면

**demo_1 CatBoost**를 선택합니다.

1. 1~38주차를 매주 처리하므로 서비스 갱신 주기와 일치합니다.
2. CatBoost는 두 버전에서 모두 PR-AUC가 XGBoost보다 높았습니다.
3. CatBoost 원시 확률의 Brier·ECE가 XGBoost보다 훨씬 안정적입니다.
4. 범주형 One-Hot이 없어 구현과 유지보수가 더 단순합니다.

### 실제 프로젝트에 권장하는 개선본

1. demo_1처럼 전체 주차 학생-과목-주차 행을 생성합니다.
2. demo2의 현재·이전·변화·최근성·상세 평가 Feature를 붙입니다.
3. CatBoost를 학생 단위 분할로 학습합니다.
4. 조기 종료용 Validation과 최종 Test를 분리합니다.
5. 별도 Calibration 또는 OOF 방식으로 Platt 보정을 검증합니다.
6. 매주 위험확률과 상위 위험군을 갱신합니다.

XGBoost는 비교 기준선으로 유지하되, 현재 원시 확률은 사용자 화면에 표시하지 않습니다.


In [ ]:
# 각 Feature 설명을 표로 다시 보고 싶을 때 실행하세요.
import pandas as pd

feature_dictionary = pd.read_csv('../demo2/feature_dictionary_demo2.csv')
feature_dictionary
